# Collect Falcon 9 launch records

IBM Applied Data Science Capstone. The checked-in course extract contains 90 Falcon 9 records. The optional refresh uses the archived IBM launch response and related SpaceX v4 resources. Related API records may change.

In [ ]:
import datetime
from functools import lru_cache
import pandas as pd
import requests

refresh_from_api = False
archive = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json'

In [ ]:
def get_json(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return response.json()

@lru_cache(maxsize=None)
def related(resource, identifier):
    if identifier is None:
        return {}
    return get_json(f'https://api.spacexdata.com/v4/{resource}/{identifier}')

def launch_record(launch):
    if len(launch['cores']) != 1 or len(launch['payloads']) != 1:
        return None
    date = datetime.date.fromisoformat(launch['date_utc'][:10])
    if date > datetime.date(2020,11,13):
        return None
    rocket = related('rockets', launch['rocket'])
    if rocket.get('name') != 'Falcon 9':
        return None
    site = related('launchpads', launch['launchpad'])
    payload = related('payloads', launch['payloads'][0])
    core = launch['cores'][0]
    history = related('cores', core['core'])
    return {'FlightNumber': launch['flight_number'], 'Date': date.isoformat(),
            'BoosterVersion': rocket['name'], 'PayloadMass': payload.get('mass_kg'),
            'Orbit': payload.get('orbit'), 'LaunchSite': site.get('name'),
            'Outcome': f"{core['landing_success']} {core['landing_type']}",
            'Flights': core['flight'], 'GridFins': core['gridfins'], 'Reused': core['reused'],
            'Legs': core['legs'], 'LandingPad': core['landpad'], 'Block': history.get('block'),
            'ReusedCount': history.get('reuse_count'), 'Serial': history.get('serial'),
            'Longitude': site.get('longitude'), 'Latitude': site.get('latitude')}

In [ ]:
if refresh_from_api:
    rows = [launch_record(launch) for launch in get_json(archive)]
    launch_df = pd.DataFrame(row for row in rows if row is not None)
    launch_df = launch_df.sort_values(['Date','FlightNumber']).reset_index(drop=True)
    launch_df['FlightNumber'] = launch_df.index + 1
    launch_df['PayloadMass'] = launch_df['PayloadMass'].fillna(launch_df['PayloadMass'].mean())
else:
    launch_df = pd.read_csv('data/dataset_part_1.csv')

launch_df[['FlightNumber','Date','Orbit','LaunchSite','Outcome']].head()

In [ ]:
launch_df[['PayloadMass', 'LandingPad', 'Outcome']].isna().sum()

## Authors


<a href="https://www.linkedin.com/in/joseph-s-50398b136/">Joseph Santarcangelo</a> has a PhD in Electrical Engineering, his research focused on using machine learning, signal processing, and computer vision to determine how videos impact human cognition. Joseph has been working for IBM since he completed his PhD. 


Copyright ©IBM Corporation. All rights reserved.
